In [ ]:
# Install dependencies once from the project root:
# python -m pip install -r requirements.txt


In [ ]:
# -*- coding: utf-8 -*-
"""Kit Digital 2022 v2 â€” enriched covariates, dual horizon (t+1/t+2), no PolicyTree.

Changes vs v1:
  1. VA panel 2019-2021 added to CORE_COVS
  2. Event study for employees (new)
  3. Outcomes: 2023 (t+1) and 2024 (t+2)
  4. No PolicyTree â€” focus on CATE/SHAP/BLP heterogeneity
  5. Output: _outputs/kd_2022_v2/
"""

from pathlib import Path

import json, pickle, warnings
import numpy as np
import pandas as pd
import shap
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import (HistGradientBoostingRegressor,
                              HistGradientBoostingClassifier,
                              GradientBoostingRegressor as _GBR)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)


In [ ]:
"""## 2. Local configuration"""

from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'panel'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'kd_2022_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SABI_OUT = DATA_DIR / 'sabi_panel.parquet'
BDNS_OUT = DATA_DIR / 'bdns_grants.parquet'

assert SABI_OUT.exists(), f'Not found: {SABI_OUT}'
assert BDNS_OUT.exists(), f'Not found: {BDNS_OUT}'
print(f'Project root: {PROJECT_ROOT}')
print(f'SABI : {SABI_OUT}')
print(f'BDNS : {BDNS_OUT}')
print(f'Output: {OUT_DIR}')


In [ ]:
# â”€â”€ Nuisance models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=1000, max_leaf_nodes=31,
    min_samples_leaf=50, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, random_state=42,
)
HIST_Y = HistGradientBoostingRegressor(**_HGB_PARAMS)
HIST_T = HistGradientBoostingClassifier(**_HGB_PARAMS)

CF_PARAMS = dict(
    model_y=HistGradientBoostingRegressor(**_HGB_PARAMS),
    model_t=HistGradientBoostingClassifier(**_HGB_PARAMS),
    discrete_treatment=True,
    n_estimators=2000, min_samples_leaf=50, max_depth=None,
    cv=5, n_jobs=-1, verbose=1, random_state=42,
)
SHAP_MAX = 5_000

In [ ]:
# â”€â”€ Load data â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sabi   = pd.read_parquet(SABI_OUT)
grants = pd.read_parquet(BDNS_OUT)
print(f'SABI: {sabi["nif"].nunique():,} firms | years {sorted(sabi["year"].unique())}')
print(f'BDNS: {grants["nif"].nunique():,} firms | years {sorted(grants["anio"].unique())}')

# VA coherence filter
abs_ratio = (sabi['valor_agregado_keur'].abs() /
             sabi['ingresos_explotacion_keur'].abs().clip(lower=0.01))
bad_va = (sabi['valor_agregado_keur'].notna() &
          sabi['ingresos_explotacion_keur'].notna() &
          (sabi['ingresos_explotacion_keur'].abs() > 0) &
          (abs_ratio > 5))
sabi.loc[bad_va, 'valor_agregado_keur'] = np.nan
print(f'VA coherence filter: {bad_va.sum():,} firm-years â†’ NaN')

In [ ]:
# â”€â”€ Treatment identification â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
conv = grants['convocatoria'].str.lower()

grants['is_kd'] = (
    conv.str.contains('kit digital', na=False) &
    conv.str.contains('segmento', na=False)
)
kd = grants.loc[grants['is_kd'], ['nif', 'anio']].drop_duplicates()
kd_yr = {yr: set(kd.loc[kd['anio'] == yr, 'nif']) for yr in [2021, 2022, 2023]}

sabi_nifs    = set(sabi['nif'].unique())
treated_nifs = (kd_yr[2022] - kd_yr[2021] - kd_yr[2023]) & sabi_nifs
control_nifs = sabi_nifs - (kd_yr[2021] | kd_yr[2022] | kd_yr[2023])

# Eligibility: PYME 1-49 emp, operative (rev > 0) in 2021
sabi_2021    = sabi[sabi['year'] == 2021]
emp_2021     = sabi_2021.set_index('nif')['num_empleados']
ing_2021     = sabi_2021.set_index('nif')['ingresos_explotacion_keur']
eligible     = set(emp_2021[(emp_2021.between(1, 49)) & (ing_2021 > 0)].index)
treated_nifs = treated_nifs & eligible
control_nifs = control_nifs & eligible
print(f'Treated={len(treated_nifs):,}  Control={len(control_nifs):,}  (after eligibility)')

In [ ]:
# â”€â”€ Subsidy amounts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
kd_amt    = grants.loc[grants['is_kd']].groupby(['nif','anio'])['ayudaEquivalente'].sum()
otras_amt = grants.loc[~grants['is_kd']].groupby(['nif','anio'])['ayudaEquivalente'].sum()

def amt(series, year):
    return (series.xs(year, level='anio')
            if year in series.index.get_level_values('anio')
            else pd.Series(dtype=float))

otras_2021  = amt(otras_amt, 2021)
kd_2022_amt = amt(kd_amt, 2022)

In [ ]:
# â”€â”€ Revenue panel 2019-2021 (log1p) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rev_panel = (sabi[sabi['year'].isin([2019,2020,2021])]
             .pivot_table(index='nif', columns='year',
                          values='ingresos_explotacion_keur', aggfunc='first')
             .rename(columns={y: f'rev_{y}' for y in [2019,2020,2021]}))
for c in rev_panel.columns:
    rev_panel[c] = np.log1p(rev_panel[c].clip(lower=0))

# â”€â”€ Employment panel 2019-2021 (log1p) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
emp_panel = (sabi[sabi['year'].isin([2019,2020,2021])]
             .pivot_table(index='nif', columns='year',
                          values='num_empleados', aggfunc='first')
             .rename(columns={y: f'emp_{y}' for y in [2019,2020,2021]}))
for y in [2019,2020,2021]:
    emp_panel[f'log_emp_{y}'] = np.log1p(emp_panel[f'emp_{y}'].clip(lower=0))
emp_panel = emp_panel[[f'log_emp_{y}' for y in [2019,2020,2021]]]

# â”€â”€ VA panel 2019-2021 (arcsinh) â€” v2: includes 2021 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
va_panel = (sabi[sabi['year'].isin([2019,2020,2021])]
            .pivot_table(index='nif', columns='year',
                         values='valor_agregado_keur', aggfunc='first')
            .rename(columns={y: f'ihs_va_{y}' for y in [2019,2020,2021]}))
for y in [2019,2020,2021]:
    va_panel[f'ihs_va_{y}'] = np.arcsinh(va_panel[f'ihs_va_{y}'])

# â”€â”€ Cross-section 2021 (firm chars, no VA â€” comes from va_panel) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
base_2021 = (sabi[sabi['year'] == 2021]
             [['nif','total_activo_keur','firm_age','nace_code','comunidad_autonoma']]
             .set_index('nif'))
base_2021['log_activo_2021']    = np.log1p(base_2021['total_activo_keur'].clip(lower=0))
base_2021['nace2']              = base_2021['nace_code'].fillna('XX').astype(str).str[:2]
base_2021['log_other_sub_2021'] = np.log1p(
    base_2021.index.to_series().map(otras_2021).fillna(0).clip(lower=0))

In [ ]:
# â”€â”€ Outcomes 2023 (t+1) and 2024 (t+2) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def build_outcomes(year):
    d = (sabi[sabi['year'] == year]
         [['nif','ingresos_explotacion_keur','valor_agregado_keur','num_empleados']]
         .rename(columns={'ingresos_explotacion_keur': f'rev_{year}',
                          'valor_agregado_keur':       f'va_{year}',
                          'num_empleados':             f'emp_{year}'})
         .set_index('nif'))
    d[f'log_rev_{year}'] = np.log1p(d[f'rev_{year}'].clip(lower=0))
    d[f'ihs_va_{year}']  = np.arcsinh(d[f'va_{year}'])
    d[f'log_emp_{year}'] = np.log1p(d[f'emp_{year}'].clip(lower=0))
    return d

outcomes_2023 = build_outcomes(2023)
outcomes_2024 = build_outcomes(2024)

In [ ]:
# â”€â”€ Master assembly â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
analysis_nifs = sorted(treated_nifs | control_nifs)
master = pd.DataFrame({'nif': analysis_nifs})
master['treated'] = master['nif'].isin(treated_nifs).astype(int)
master = (master.set_index('nif')
          .join(base_2021,     how='left')
          .join(rev_panel,     how='left')
          .join(emp_panel,     how='left')
          .join(va_panel,      how='left')
          .join(outcomes_2023, how='left')
          .join(outcomes_2024, how='left')
          .reset_index())
master['kd_keur_2022'] = master['nif'].map(kd_2022_amt).fillna(0.0)
print(f'Master: {len(master):,} | treated={master["treated"].sum():,} | '
      f'control={(master["treated"]==0).sum():,}')

In [ ]:
# â”€â”€ Sector & region encoding â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
top_nace = master['nace2'].value_counts().head(19).index.tolist()
master['nace2_grp'] = master['nace2'].where(master['nace2'].isin(top_nace), 'OT')

NUTS1_MAP = {
    'Galicia':'ES1_Noroeste','Asturias':'ES1_Noroeste','Cantabria':'ES1_Noroeste',
    'PaÃ­s Vasco':'ES2_Noreste','Navarra':'ES2_Noreste','La Rioja':'ES2_Noreste',
    'AragÃ³n':'ES2_Noreste','Madrid':'ES3_Madrid',
    'Castilla y LeÃ³n':'ES4_Centro','Castilla-La Mancha':'ES4_Centro','Extremadura':'ES4_Centro',
    'CataluÃ±a':'ES5_Este','Comunidad Valenciana':'ES5_Este','Baleares':'ES5_Este',
    'AndalucÃ­a':'ES6_Sur','Murcia':'ES6_Sur','Ceuta':'ES6_Sur','Melilla':'ES6_Sur',
    'Canarias':'ES7_Canarias',
}
master['nuts1'] = master['comunidad_autonoma'].fillna('Unknown').map(NUTS1_MAP).fillna('Unknown')
nace_d  = pd.get_dummies(master['nace2_grp'], prefix='nace',  drop_first=True)
nuts1_d = pd.get_dummies(master['nuts1'],     prefix='nuts1', drop_first=True)
master  = pd.concat([master, nace_d, nuts1_d], axis=1)
CAT_COLS = list(nace_d.columns) + list(nuts1_d.columns)

In [ ]:
# â”€â”€ Feature matrix â€” v2 adds VA panel to CORE_COVS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CORE_COVS = [
    'rev_2019', 'rev_2020', 'rev_2021',
    'log_emp_2019', 'log_emp_2020', 'log_emp_2021',
    'ihs_va_2019', 'ihs_va_2020', 'ihs_va_2021',   # NEW in v2
    'log_activo_2021', 'log_other_sub_2021', 'firm_age',
]
ALL_COVS = CORE_COVS + CAT_COLS
CONT_COLS = [c for c in ALL_COVS if c not in CAT_COLS]
print(f'Feature matrix: {len(ALL_COVS)} cols '
      f'({len(CORE_COVS)} continuous + {len(CAT_COLS)} dummies)')

print('\nComplete-case preview (with enriched CORE_COVS):')
for outcome, label in [('log_rev_2023','rev_y1'), ('log_rev_2024','rev_y2'),
                        ('ihs_va_2023','va_y1'),  ('ihs_va_2024','va_y2'),
                        ('log_emp_2023','emp_y1'), ('log_emp_2024','emp_y2')]:
    cc = master[list(dict.fromkeys(ALL_COVS + [outcome,'treated']))].dropna()
    t  = int(cc['treated'].sum())
    print(f'  {label:<10}: N={len(cc):,}  treated={t:,}  control={len(cc)-t:,}')

In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# HELPERS
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

def ps_trim(master_df, outcome_col, extra_cols=None, label='', lo=0.05, hi=0.95):
    """Complete-case + PS trim. Returns trimmed DataFrame."""
    extra_cols = extra_cols or []
    cols = list(dict.fromkeys(ALL_COVS + extra_cols + [outcome_col, 'treated', 'nif']))
    d  = master_df[cols].dropna().copy()
    Xp = d[ALL_COVS].values.astype('float32')
    Tp = d['treated'].values
    ps = cross_val_predict(HIST_T, Xp, Tp, cv=5, method='predict_proba')[:, 1]
    keep = (ps >= lo) & (ps <= hi)
    d  = d[keep].copy()
    PS_TRIM_ROWS.append({
        'specification': label,
        'outcome_col': outcome_col,
        'complete_n': int(len(ps)),
        'complete_treated': int(Tp.sum()),
        'complete_control': int(len(Tp) - Tp.sum()),
        'trimmed_n': int(len(d)),
        'trimmed_treated': int(d['treated'].sum()),
        'trimmed_control': int((d['treated'] == 0).sum()),
        'kept_share': float(keep.mean()),
        'trim_lo': lo,
        'trim_hi': hi,
    })
    print(f'  [{label}] N={len(d):,}  treated={int(d["treated"].sum()):,}  '
          f'control={int((d["treated"]==0).sum()):,}  '
          f'({keep.mean():.1%} kept)')
    return d


def scale_and_extract(d, outcome_col):
    """Scale continuous covariates; return Y, T, X, nifs."""
    d = d.copy()
    d[CONT_COLS] = StandardScaler().fit_transform(d[CONT_COLS])
    return (d[outcome_col].values,
            d['treated'].values,
            d[ALL_COVS].values.astype('float32'),
            d['nif'].values)



# Publication labels for plots only. Internal column names stay unchanged.
FEATURE_LABELS = {
    'rev_2019': 'Log operating revenues (2019)',
    'rev_2020': 'Log operating revenues (2020)',
    'rev_2021': 'Log operating revenues (2021)',
    'log_emp_2019': 'Log employment (2019)',
    'log_emp_2020': 'Log employment (2020)',
    'log_emp_2021': 'Log employment (2021)',
    'ihs_va_2019': 'IHS value added (2019)',
    'ihs_va_2020': 'IHS value added (2020)',
    'ihs_va_2021': 'IHS value added (2021)',
    'log_activo_2021': 'Log assets (2021)',
    'log_other_sub_2021': 'Log other subsidies (2021)',
    'firm_age': 'Firm age',
    'nace_10': 'Sector: NACE 10 food manufacturing',
    'nace_25': 'Sector: NACE 25 fabricated metals',
    'nace_28': 'Sector: NACE 28 machinery',
    'nace_41': 'Sector: NACE 41 building construction',
    'nace_43': 'Sector: NACE 43 specialised construction',
    'nace_45': 'Sector: NACE 45 motor trade and repair',
    'nace_46': 'Sector: NACE 46 wholesale trade',
    'nace_47': 'Sector: NACE 47 retail trade',
    'nace_49': 'Sector: NACE 49 land transport',
    'nace_55': 'Sector: NACE 55 accommodation',
    'nace_56': 'Sector: NACE 56 food services',
    'nace_62': 'Sector: NACE 62 computer programming',
    'nace_68': 'Sector: NACE 68 real estate',
    'nace_69': 'Sector: NACE 69 legal and accounting',
    'nace_71': 'Sector: NACE 71 architecture and engineering',
    'nace_73': 'Sector: NACE 73 advertising and market research',
    'nace_85': 'Sector: NACE 85 education',
    'nace_86': 'Sector: NACE 86 human health',
    'nace_OT': 'Sector: other NACE groups',
    'nuts1_ES2_Noreste': 'Region: Northeast',
    'nuts1_ES3_Madrid': 'Region: Madrid',
    'nuts1_ES4_Centro': 'Region: Centre',
    'nuts1_ES5_Este': 'Region: East',
    'nuts1_ES6_Sur': 'Region: South',
    'nuts1_ES7_Canarias': 'Region: Canary Islands',
    'nuts1_Unknown': 'Region: unknown',
}

OUTCOME_LABELS = {
    'Revenues': 'Log operating revenues',
    'VA': 'IHS value added',
    'Employment': 'Log employment',
}

DUMMY_PREFIXES = ('nace_', 'nuts1_')
SHAP_PLOT_DATA = {}


# Structured output stores used to populate manuscript tables after a clean run.
PS_TRIM_ROWS = []
BLP_CALIBRATION_ROWS = []
BLP_MODERATOR_ROWS = []
GATES_ROWS = []

def _two_sided_p_from_z(z):
    return float(__import__('math').erfc(abs(float(z)) / np.sqrt(2))) if np.isfinite(z) else np.nan

def _linear_dml_coef_rows(est, label, feature_names, target_rows, table):
    """Append LinearDML coefficient rows in a manuscript-ready structure."""
    point = np.asarray(est.coef_).reshape(-1)
    try:
        inf = est.coef__inference()
        stderr = np.asarray(inf.stderr).reshape(-1)
        ci = inf.conf_int()
        ci_low = np.asarray(ci[0]).reshape(-1)
        ci_high = np.asarray(ci[1]).reshape(-1)
    except Exception:
        stderr = np.full_like(point, np.nan, dtype=float)
        ci_low = np.full_like(point, np.nan, dtype=float)
        ci_high = np.full_like(point, np.nan, dtype=float)

    for name, pt, se, lo, hi in zip(feature_names, point, stderr, ci_low, ci_high):
        z = float(pt / se) if np.isfinite(se) and se != 0 else np.nan
        target_rows.append({
            'table': table,
            'specification': label,
            'feature': name,
            'feature_label': feature_label(name),
            'estimate': float(pt),
            'std_error': float(se) if np.isfinite(se) else np.nan,
            'z': z,
            'p_value': _two_sided_p_from_z(z),
            'ci_low': float(lo) if np.isfinite(lo) else np.nan,
            'ci_high': float(hi) if np.isfinite(hi) else np.nan,
        })

def export_table_outputs():
    """Write structured tables used by the manuscript."""
    if BLP_CALIBRATION_ROWS:
        pd.DataFrame(BLP_CALIBRATION_ROWS).to_csv(OUT_DIR / 'table_calibration.csv', index=False)
    if BLP_MODERATOR_ROWS:
        pd.DataFrame(BLP_MODERATOR_ROWS).to_csv(OUT_DIR / 'table_blp_moderators.csv', index=False)
    if GATES_ROWS:
        pd.DataFrame(GATES_ROWS).to_csv(OUT_DIR / 'table_gates.csv', index=False)
    if PS_TRIM_ROWS:
        pd.DataFrame(PS_TRIM_ROWS).to_csv(OUT_DIR / 'table_sample_counts.csv', index=False)

def feature_label(name):
    return FEATURE_LABELS.get(name, name.replace('_', ' '))

def feature_labels(names):
    return [feature_label(name) for name in names]

def plot_feature_label(name):
    label = feature_label(name)
    for year in ('2019', '2020', '2021'):
        label = label.replace(f' ({year})', '')
    return label

def plot_feature_labels(names):
    return [plot_feature_label(name) for name in names]

def outcome_label(short_name):
    return OUTCOME_LABELS.get(short_name, short_name)

def compute_shap(model, X, feature_names, label, out_path):
    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(X), size=min(SHAP_MAX, len(X)), replace=False)
    Xs   = X[idx]
    cs   = np.squeeze(model.effect(Xs))
    surr = _GBR(n_estimators=300, max_depth=3, learning_rate=0.05,
                min_samples_leaf=20, random_state=42)
    surr.fit(Xs, cs)
    expl = shap.TreeExplainer(surr)
    sv   = expl.shap_values(Xs, check_additivity=False)
    df   = pd.DataFrame(sv, columns=feature_names)
    xdf  = pd.DataFrame(Xs, columns=feature_names)
    df.to_parquet(out_path, index=False)
    xdf.to_parquet(out_path.with_name(out_path.stem + '_features.parquet'), index=False)
    SHAP_PLOT_DATA[label] = {
        'shap': df,
        'features': xdf,
        'feature_names': list(feature_names),
        'display_names': plot_feature_labels(feature_names),
    }
    slug = label.lower().replace(' ','_').replace('(','').replace(')','').replace('+','p')
    display_names = plot_feature_labels(feature_names)
    shap.summary_plot(sv, Xs, feature_names=display_names, max_display=15, show=False)
    plt.title(f'SHAP - {label}', fontsize=10); plt.tight_layout()
    plt.savefig(OUT_DIR / f'fig_shap_beeswarm_{slug}.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    shap.summary_plot(sv, Xs, feature_names=display_names,
                      plot_type='bar', max_display=15, show=False)
    plt.title(f'SHAP importance - {label}', fontsize=10); plt.tight_layout()
    plt.savefig(OUT_DIR / f'fig_shap_bar_{slug}.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  SHAP [{label}]: shape={df.shape}  saved')
    return df


def run_forest(Y, T, X, nifs, pkl_name, label):
    """Train or load CausalForestDML; save ATE + CATE; return (model, ate_d, cate, shap_df)."""
    pkl_path = OUT_DIR / pkl_name
    if pkl_path.exists():
        with open(pkl_path, 'rb') as f:
            model = pickle.load(f)
        print(f'  Loaded: {pkl_path.name}')
    else:
        print(f'  Training {label}...')
        model = CausalForestDML(**CF_PARAMS)
        model.fit(Y, T, X=X)
        with open(pkl_path, 'wb') as f:
            pickle.dump(model, f)
        print(f'  Saved: {pkl_path.name}')

    stem    = pkl_name.replace('.pkl', '')
    ate_inf = model.ate_inference(X)
    ate_d   = {
        'ate':    float(np.squeeze(ate_inf.mean_point)),
        'ate_se': float(np.squeeze(ate_inf.stderr_mean)),
        'ate_lb': float(np.squeeze(ate_inf.conf_int_mean()[0])),
        'ate_ub': float(np.squeeze(ate_inf.conf_int_mean()[1])),
        'n':      int(len(Y)),
    }
    (OUT_DIR / f'{stem}_ate.json').write_text(json.dumps(ate_d, indent=2))

    cate       = np.squeeze(model.effect(X))
    lb, ub     = model.effect_interval(X, alpha=0.05)
    lb, ub     = np.squeeze(lb), np.squeeze(ub)
    cate_df    = pd.DataFrame({'nif': nifs, 'cate': cate, 'cate_lb': lb, 'cate_ub': ub})
    cate_df.to_parquet(OUT_DIR / f'{stem}_cate.parquet', index=False)

    print(f'  ATE={ate_d["ate"]:+.4f}  SE={ate_d["ate_se"]:.4f}  '
          f'95%CI [{ate_d["ate_lb"]:+.4f}, {ate_d["ate_ub"]:+.4f}]  N={ate_d["n"]:,}')
    print(f'  CATE mean={cate.mean():+.4f}  SD={cate.std():.4f}  '
          f'P25={np.percentile(cate,25):+.4f}  P75={np.percentile(cate,75):+.4f}  '
          f'%pos={(cate>0).mean():.1%}')

    shap_df = compute_shap(model, X, ALL_COVS, label, OUT_DIR / f'{stem}_shap.parquet')
    return model, ate_d, cate, shap_df


HET_MODS = ['log_emp_2021', 'rev_2021', 'ihs_va_2021', 'log_activo_2021',
            'firm_age', 'log_other_sub_2021']

def run_blp(Y, T, X, cate, label):
    print(f'\n  --- BLP [{label}] ---')
    # Calibration
    cate_c  = (cate - cate.mean()).reshape(-1, 1)
    cal_blp = LinearDML(model_y=HIST_Y, model_t=HIST_T,
                        discrete_treatment=True, cv=5, random_state=42)
    cal_blp.fit(Y, T, X=cate_c, W=X)
    print('  Calibration (differential_pred should be ~1 if forest is unbiased):')
    print(cal_blp.summary(feature_names=['differential_pred']))
    _linear_dml_coef_rows(
        cal_blp, label, ['differential_pred'], BLP_CALIBRATION_ROWS, 'calibration')
    # Heterogeneity moderators
    het_feats = [f for f in HET_MODS if f in ALL_COVS]
    het_idx   = [ALL_COVS.index(f) for f in het_feats]
    Xh = X[:, het_idx]
    W  = np.delete(X, het_idx, axis=1)
    het_blp = LinearDML(model_y=HIST_Y, model_t=HIST_T,
                        discrete_treatment=True, cv=5, random_state=42)
    het_blp.fit(Y, T, X=Xh, W=W)
    print('  Heterogeneity moderators:')
    print(het_blp.summary(feature_names=het_feats))
    _linear_dml_coef_rows(
        het_blp, label, het_feats, BLP_MODERATOR_ROWS, 'blp_moderators')
    return {'calibration': cal_blp, 'moderators': het_blp}


def run_gates(model, X, nifs, label):
    emp_vals = master.set_index('nif').loc[nifs, 'log_emp_2021'].fillna(0).values
    emp_lvl  = np.expm1(emp_vals)
    seg      = np.select([emp_lvl < 3, emp_lvl < 10],
                         ['III (<3)', 'II (3-9)'], 'I (10-49)')
    print(f'\n  --- GATES by segment [{label}] ---')
    for s in ['III (<3)', 'II (3-9)', 'I (10-49)']:
        m = seg == s
        if m.sum() < 100:
            continue
        gi = model.ate_inference(X[m])
        pt = float(np.squeeze(gi.mean_point))
        se = float(np.squeeze(gi.stderr_mean))
        ci_low, ci_high = pt - 1.96 * se, pt + 1.96 * se
        GATES_ROWS.append({
            'specification': label,
            'segment': s,
            'n': int(m.sum()),
            'gate': pt,
            'std_error': se,
            'ci_low': ci_low,
            'ci_high': ci_high,
        })
        print(f'  Seg {s:<12} n={m.sum():>6,}: GATE={pt:+.4f}  SE={se:.4f}  '
              f'[{ci_low:+.4f}, {ci_high:+.4f}]')


In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PHASE 1 â€” PS OVERLAP + DIAGNOSTICS
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print('\n' + '='*65 + '\nPS OVERLAP â€” revenues (reference)\n' + '='*65)
_req = list(dict.fromkeys(ALL_COVS + ['log_rev_2023', 'treated', 'nif']))
_d   = master[_req].dropna().copy()
_Xb  = _d[ALL_COVS].values.astype('float32')
_Tb  = _d['treated'].values
_ps  = cross_val_predict(HIST_T, _Xb, _Tb, cv=5, method='predict_proba')[:, 1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(_ps[_Tb==0], bins=50, alpha=0.5, label='Control', density=True, color='steelblue')
ax.hist(_ps[_Tb==1], bins=50, alpha=0.5, label='Tratado', density=True, color='darkorange')
ax.set_xlabel(r'$\hat{m}(X)$'); ax.set_title('PS Overlap â€” KD 2022 v2'); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_ps_overlap.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)

from sklearn.inspection import permutation_importance
_gbc = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500, random_state=42)
_gbc.fit(_Xb, _Tb)
_auc = roc_auc_score(_Tb, _gbc.predict_proba(_Xb)[:, 1])
print(f'AUC PS = {_auc:.4f}')

_keep   = (_ps >= 0.05) & (_ps <= 0.95)
_nifs_b = _d['nif'].values[_keep]
print(f'Trimmed: {_keep.sum():,} ({_keep.mean():.1%})')

# LinearDML revenues t+1 (reference spec)
_Xb2 = _Xb[_keep]; _Yb = _d['log_rev_2023'].values[_keep]; _Tb2 = _Tb[_keep]
_ld  = LinearDML(model_y=HIST_Y, model_t=HIST_T, discrete_treatment=True, cv=5, random_state=42)
_ld.fit(_Yb, _Tb2, X=None, W=_Xb2)
print('\nLinearDML revenues t+1 (constant-effect benchmark):')
print(_ld.summary())

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ---- journal style (set once) ------------------------------------------------
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.linewidth": 0.7, "axes.edgecolor": "#333333",
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
NAVY, GREY = "#1f3b63", "#9aa7b4"
lo, hi = 0.05, 0.95

# ---- your real quantities ----------------------------------------------------
# _ps   : cross-fitted propensity scores  (you already compute this)
# _Tb   : treatment indicator
# _auc  : roc_auc_score(...)              (you already compute this)
keep_frac = ((_ps >= lo) & (_ps <= hi)).mean()

fig, ax = plt.subplots(figsize=(6.0, 3.4))
bins = np.linspace(0, 1, 46)
ax.hist(_ps[_Tb == 0], bins=bins, density=True, color=GREY, alpha=0.85,
        edgecolor="white", linewidth=0.3, label="Control", zorder=2)
ax.hist(_ps[_Tb == 1], bins=bins, density=True, histtype="stepfilled",
        facecolor="none", edgecolor=NAVY, linewidth=1.4,
        label="Treated", zorder=3)

for b in (lo, hi):
    ax.axvline(b, color="#555555", linestyle=(0, (4, 3)), linewidth=0.8, zorder=1)
ax.axvspan(0, lo, color="#000000", alpha=0.04, zorder=0)
ax.axvspan(hi, 1, color="#000000", alpha=0.04, zorder=0)

ymax = ax.get_ylim()[1]
ax.text(lo + 0.008, ymax*0.06, "trim 0.05", ha="left",  va="bottom",
        fontsize=7.5, color="#555555", style="italic")
ax.text(hi - 0.008, ymax*0.06, "trim 0.95", ha="right", va="bottom",
        fontsize=7.5, color="#555555", style="italic")

ax.text(0.985, 0.97,
        f"AUC = {_auc:.3f}\nRetained after trimming = {keep_frac*100:.1f}%",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#cccccc", linewidth=0.6))

ax.set_xlabel(r"Estimated propensity score $\hat{m}(X)$")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)
ax.xaxis.set_major_locator(MultipleLocator(0.2))
ax.xaxis.set_minor_locator(MultipleLocator(0.05))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper left", fontsize=9,
          handlelength=1.4, borderaxespad=0.4)

fig.tight_layout()
fig.savefig(OUT_DIR / "fig_ps_overlap.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PHASE 2 â€” EVENT STUDIES (revenues, VA, employees)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•


def event_study(nifs_sample, ev_years, base_year, stable_controls,
                col_fn, label, out_name):
    """Run event study; return relative gaps, standard errors, and a table."""
    gaps, ses, ns = {}, {}, {}
    for t in ev_years:
        col = col_fn(t)
        dd  = master[master['nif'].isin(nifs_sample)][[col] + stable_controls + ['treated']].dropna()
        est = LinearDML(model_y=HIST_Y, model_t=HIST_T,
                        discrete_treatment=True, cv=5, random_state=42)
        est.fit(dd[col].values, dd['treated'].values,
                X=None, W=dd[stable_controls].values.astype('float32'))
        gaps[t] = float(np.squeeze(est.intercept_))
        ses[t]  = float(np.squeeze(est.intercept__inference().stderr))
        ns[t]   = int(len(dd))
    g0  = gaps[base_year]
    rel = {t: gaps[t] - g0 for t in ev_years}
    rows = []
    for t in ev_years:
        rows.append({
            'outcome': label,
            'year': int(t),
            'base_year': int(base_year),
            'estimate': float(gaps[t]),
            'relative_gap': float(rel[t]),
            'std_error': float(ses[t]),
            'ci_low': float(rel[t] - 1.96 * ses[t]),
            'ci_high': float(rel[t] + 1.96 * ses[t]),
            'n': ns[t],
        })

    fig = plt.figure(figsize=(7.5, 4.2))
    plt.axhline(0, color='gray', lw=0.8, ls='--')
    plt.axvspan(base_year + 0.5, max(ev_years) + 0.5, color='crimson', alpha=0.06)
    plt.errorbar(ev_years, [rel[t] for t in ev_years],
                 yerr=[1.96*ses[t] for t in ev_years],
                 fmt='o-', capsize=4, color='steelblue')
    plt.title(f'Event study - {label} (KD 2022 v2)')
    plt.xlabel('Year'); plt.ylabel(f'Gap vs {base_year}')
    plt.tight_layout()
    plt.savefig(OUT_DIR / out_name, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'\nEvent study [{label}]:')
    for t in ev_years:
        print(f'  {t}: {rel[t]:+.4f}  '
              f'[{rel[t]-1.96*ses[t]:+.4f}, {rel[t]+1.96*ses[t]:+.4f}]')
    return rel, ses, pd.DataFrame(rows)

# Revenues event study
print('\n' + '='*65 + '\nEVENT STUDY â€” Revenues\n' + '='*65)
_stable_rev = ['log_activo_2021','firm_age','log_other_sub_2021',
               'ihs_va_2021','log_emp_2021'] + CAT_COLS
rel_r, ses_r, es_r_df = event_study(
    _nifs_b, [2019, 2020, 2021, 2023, 2024], 2021, _stable_rev,
    lambda t: 'log_rev_2023' if t == 2023 else 'log_rev_2024' if t == 2024 else f'rev_{t}',
    'Revenues', 'fig_event_study_revenues.pdf')

# VA event study â€” use VA-trimmed sample
print('\n' + '='*65 + '\nEVENT STUDY â€” Valor Agregado\n' + '='*65)
_cols_va_es = list(dict.fromkeys(ALL_COVS + ['ihs_va_2023','treated','nif']))
_dva_es = master[_cols_va_es].dropna().copy()
_ps_va  = cross_val_predict(HIST_T, _dva_es[ALL_COVS].values.astype('float32'),
                             _dva_es['treated'].values, cv=5, method='predict_proba')[:, 1]
_dva_es = _dva_es[(_ps_va >= 0.05) & (_ps_va <= 0.95)].copy()
_va23   = np.sinh(_dva_es['ihs_va_2023'].values)
_va21   = np.sinh(_dva_es['ihs_va_2021'].values)
_dva_es = _dva_es[(_va23 > 0) & (_va21 > 0)].copy()
_nifs_va_es = _dva_es['nif'].values
print(f'VA event-study sample: N={len(_dva_es):,}')

_stable_va = ['log_activo_2021','firm_age','log_other_sub_2021',
              'rev_2021','log_emp_2021'] + CAT_COLS
rel_va, ses_va, es_va_df = event_study(
    _nifs_va_es, [2019, 2020, 2021, 2023, 2024], 2021, _stable_va,
    lambda t: 'ihs_va_2023' if t == 2023 else 'ihs_va_2024' if t == 2024 else f'ihs_va_{t}',
    'Valor Agregado', 'fig_event_study_va.pdf')

# Employees event study (NEW in v2)
print('\n' + '='*65 + '\nEVENT STUDY â€” Employees (NEW)\n' + '='*65)
_cols_emp_es = list(dict.fromkeys(ALL_COVS + ['log_emp_2023','treated','nif']))
_demp_es = master[_cols_emp_es].dropna().copy()
_ps_emp  = cross_val_predict(HIST_T, _demp_es[ALL_COVS].values.astype('float32'),
                              _demp_es['treated'].values, cv=5, method='predict_proba')[:, 1]
_demp_es = _demp_es[(_ps_emp >= 0.05) & (_ps_emp <= 0.95)].copy()
_nifs_emp_es = _demp_es['nif'].values
print(f'Emp event-study sample: N={len(_demp_es):,}')

# Stable controls exclude emp lags (those are the outcomes in the event study)
_stable_emp = ['log_activo_2021','firm_age','log_other_sub_2021',
               'rev_2021','ihs_va_2021'] + CAT_COLS
# Employees event study
rel_emp, ses_emp, es_emp_df = event_study(
    _nifs_emp_es, [2019, 2020, 2021, 2023, 2024], 2021, _stable_emp,
    lambda t: 'log_emp_2023' if t == 2023 else 'log_emp_2024' if t == 2024 else f'log_emp_{t}',
    'Employees', 'fig_event_study_employees.pdf')

# Combined 3-panel event study
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
for ax, (lbl, rel, ses, color) in zip(axes, [
    ('Revenues',     rel_r,   ses_r,   'steelblue'),
    ('Valor Agreg.', rel_va,  ses_va,  'darkorange'),
    ('Employees',    rel_emp, ses_emp, 'forestgreen'),
]):
    yrs = sorted(rel.keys())
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.axvspan(2021.5, max(yrs)+0.5, color='crimson', alpha=0.06)
    ax.errorbar(yrs, [rel[t] for t in yrs],
                yerr=[1.96*ses[t] for t in yrs],
                fmt='o-', capsize=4, color=color)
    ax.set_title(lbl, fontsize=11); ax.set_xlabel('AÃ±o'); ax.set_ylabel('Gap vs 2021')
plt.suptitle('Parallel trends â€” KD 2022 v2', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_event_study_combined.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)
pd.concat([es_r_df, es_va_df, es_emp_df], ignore_index=True).to_csv(
    OUT_DIR / 'table_event_study.csv', index=False)
print('\nCombined event study figure saved.')


In [ ]:
# â”€â”€ Combined 3-panel event study (publication style) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.linewidth": 0.7, "axes.edgecolor": "#333333",
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
NAVY = "#1f3b63"

panels = [
    ("Revenues",     rel_r,   ses_r),
    ("Value added",  rel_va,  ses_va),
    ("Employment",   rel_emp, ses_emp),
]

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2), sharex=True)

for ax, (lbl, rel, ses) in zip(axes, panels):
    yrs = sorted(rel.keys())
    g   = [rel[t] for t in yrs]
    err = [1.96 * ses[t] for t in yrs]

    # post-treatment shading (everything after the 2021 base year)
    ax.axvspan(2021.5, max(yrs) + 0.5, color="#000000", alpha=0.04, zorder=0)
    ax.axhline(0, color="#555555", lw=0.8, ls=(0, (4, 3)), zorder=1)

    ax.errorbar(yrs, g, yerr=err, fmt="o-", color=NAVY,
                ecolor=NAVY, elinewidth=1.0, capsize=3, capthick=1.0,
                markersize=4, markerfacecolor=NAVY, markeredgecolor=NAVY,
                linewidth=1.2, zorder=3)

    ax.set_title(lbl, fontsize=10, pad=6)
    ax.set_xlabel("Year")
    ax.set_xticks(yrs)
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.08)

axes[0].set_ylabel("Gap relative to 2021")

fig.tight_layout()
fig.savefig(OUT_DIR / "fig_event_study_combined.pdf", bbox_inches="tight")
plt.close(fig)
print("Combined event study figure saved.")

In [ ]:
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# PHASE 3 â€” 6 CAUSAL FOREST MODELS
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

# â”€â”€ Revenues t+1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nREVENUES t+1 (2023)\n' + '='*65)
d_r1 = ps_trim(master, 'log_rev_2023', label='REV_Y1')
Y_r1, T_r1, X_r1, nifs_r1 = scale_and_extract(d_r1, 'log_rev_2023')
mod_r1, ate_r1, cate_r1, shap_r1 = run_forest(
    Y_r1, T_r1, X_r1, nifs_r1, 'kd22v2_revenues_y1.pkl', 'Revenues t+1 (2023)')
run_blp(Y_r1, T_r1, X_r1, cate_r1, 'Revenues t+1')
run_gates(mod_r1, X_r1, nifs_r1, 'Revenues t+1')

# â”€â”€ Revenues t+2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nREVENUES t+2 (2024)\n' + '='*65)
d_r2 = ps_trim(master, 'log_rev_2024', label='REV_Y2')
Y_r2, T_r2, X_r2, nifs_r2 = scale_and_extract(d_r2, 'log_rev_2024')
mod_r2, ate_r2, cate_r2, shap_r2 = run_forest(
    Y_r2, T_r2, X_r2, nifs_r2, 'kd22v2_revenues_y2.pkl', 'Revenues t+2 (2024)')
run_blp(Y_r2, T_r2, X_r2, cate_r2, 'Revenues t+2')
run_gates(mod_r2, X_r2, nifs_r2, 'Revenues t+2')

# â”€â”€ VA t+1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nVALOR AGREGADO t+1 (2023)\n' + '='*65)
d_v1 = ps_trim(master, 'ihs_va_2023', label='VA_Y1')
_va23 = np.sinh(d_v1['ihs_va_2023'].values); _va21 = np.sinh(d_v1['ihs_va_2021'].values)
_pos  = (_va23 > 0) & (_va21 > 0)
d_v1  = d_v1[_pos].copy()
PS_TRIM_ROWS[-1].update({
    'trimmed_n': int(len(d_v1)),
    'trimmed_treated': int(d_v1['treated'].sum()),
    'trimmed_control': int((d_v1['treated'] == 0).sum()),
    'kept_share': float(len(d_v1) / PS_TRIM_ROWS[-1]['complete_n']),
    'positive_va_filter': True,
})
print(f'VA t+1 (VA>0 filter): N={len(d_v1):,}  treated={int(d_v1["treated"].sum()):,}')
Y_v1 = np.log(_va23[_pos])
T_v1 = d_v1['treated'].values
d_v1[CONT_COLS] = StandardScaler().fit_transform(d_v1[CONT_COLS])
X_v1 = d_v1[ALL_COVS].values.astype('float32')
nifs_v1 = d_v1['nif'].values

mod_v1, ate_v1, cate_v1, shap_v1 = run_forest(
    Y_v1, T_v1, X_v1, nifs_v1, 'kd22v2_va_y1.pkl', 'VA t+1 (2023)')
run_blp(Y_v1, T_v1, X_v1, cate_v1, 'VA t+1')
run_gates(mod_v1, X_v1, nifs_v1, 'VA t+1')

# â”€â”€ VA t+2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nVALOR AGREGADO t+2 (2024)\n' + '='*65)
d_v2 = ps_trim(master, 'ihs_va_2024', label='VA_Y2')
_va24 = np.sinh(d_v2['ihs_va_2024'].values); _va21b = np.sinh(d_v2['ihs_va_2021'].values)
_pos2 = (_va24 > 0) & (_va21b > 0)
d_v2  = d_v2[_pos2].copy()
PS_TRIM_ROWS[-1].update({
    'trimmed_n': int(len(d_v2)),
    'trimmed_treated': int(d_v2['treated'].sum()),
    'trimmed_control': int((d_v2['treated'] == 0).sum()),
    'kept_share': float(len(d_v2) / PS_TRIM_ROWS[-1]['complete_n']),
    'positive_va_filter': True,
})
print(f'VA t+2 (VA>0 filter): N={len(d_v2):,}  treated={int(d_v2["treated"].sum()):,}')
Y_v2 = np.log(_va24[_pos2])
T_v2 = d_v2['treated'].values
d_v2[CONT_COLS] = StandardScaler().fit_transform(d_v2[CONT_COLS])
X_v2 = d_v2[ALL_COVS].values.astype('float32')
nifs_v2 = d_v2['nif'].values

mod_v2, ate_v2, cate_v2, shap_v2 = run_forest(
    Y_v2, T_v2, X_v2, nifs_v2, 'kd22v2_va_y2.pkl', 'VA t+2 (2024)')
run_blp(Y_v2, T_v2, X_v2, cate_v2, 'VA t+2')
run_gates(mod_v2, X_v2, nifs_v2, 'VA t+2')

# â”€â”€ Employment t+1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nEMPLOYMENT t+1 (2023)\n' + '='*65)
d_e1 = ps_trim(master, 'log_emp_2023', label='EMP_Y1')
Y_e1, T_e1, X_e1, nifs_e1 = scale_and_extract(d_e1, 'log_emp_2023')
mod_e1, ate_e1, cate_e1, shap_e1 = run_forest(
    Y_e1, T_e1, X_e1, nifs_e1, 'kd22v2_emp_y1.pkl', 'Employment t+1 (2023)')
run_blp(Y_e1, T_e1, X_e1, cate_e1, 'Employment t+1')
run_gates(mod_e1, X_e1, nifs_e1, 'Employment t+1')

# â”€â”€ Employment t+2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*65 + '\nEMPLOYMENT t+2 (2024)\n' + '='*65)
d_e2 = ps_trim(master, 'log_emp_2024', label='EMP_Y2')
Y_e2, T_e2, X_e2, nifs_e2 = scale_and_extract(d_e2, 'log_emp_2024')
mod_e2, ate_e2, cate_e2, shap_e2 = run_forest(
    Y_e2, T_e2, X_e2, nifs_e2, 'kd22v2_emp_y2.pkl', 'Employment t+2 (2024)')
run_blp(Y_e2, T_e2, X_e2, cate_e2, 'Employment t+2')
run_gates(mod_e2, X_e2, nifs_e2, 'Employment t+2')


In [ ]:
# Summary tables for ATE and CATE distributions
summary_rows = []
for name, outcome_label_full, a1, c1, a2, c2 in [
    ('Revenues', 'Log operating revenues', ate_r1, cate_r1, ate_r2, cate_r2),
    ('Value added', 'IHS value added', ate_v1, cate_v1, ate_v2, cate_v2),
    ('Employment', 'Log employment', ate_e1, cate_e1, ate_e2, cate_e2),
]:
    for horizon, ad, carr in [('t+1 (2023)', a1, c1), ('t+2 (2024)', a2, c2)]:
        z = ad['ate'] / ad['ate_se'] if ad['ate_se'] else np.nan
        summary_rows.append({
            'outcome': name,
            'outcome_label': outcome_label_full,
            'horizon': horizon,
            'ate': ad['ate'],
            'ate_se': ad['ate_se'],
            'ate_z': z,
            'ate_p': _two_sided_p_from_z(z),
            'ate_ci_low': ad['ate_lb'],
            'ate_ci_high': ad['ate_ub'],
            'n': ad['n'],
            'cate_mean': float(np.mean(carr)),
            'cate_sd': float(np.std(carr)),
            'cate_p25': float(np.percentile(carr, 25)),
            'cate_p75': float(np.percentile(carr, 75)),
            'cate_positive_share': float((carr > 0).mean()),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / 'table_ate_cate_summary.csv', index=False)
summary_df[['outcome', 'horizon', 'ate', 'ate_se', 'ate_z', 'ate_p',
            'ate_ci_low', 'ate_ci_high', 'n', 'cate_positive_share']].to_csv(
    OUT_DIR / 'table_ate.csv', index=False)
summary_df[['outcome', 'horizon', 'cate_mean', 'cate_sd', 'cate_p25',
            'cate_p75', 'cate_positive_share', 'n']].to_csv(
    OUT_DIR / 'table_cate.csv', index=False)
export_table_outputs()

print('\n' + '='*80)
print('SUMMARY - KD 2022 v2')
print('='*80)
print(f"{'Outcome':<20}{'Horizon':<14}{'ATE':>8}{'SE':>8}{'CI_lo':>10}"
      f"{'CI_hi':>10}{'N':>8}{'%pos':>8}")
print('-'*86)
for _, row in summary_df.iterrows():
    print(f"  {row['outcome']:<18}{row['horizon']:<14}{row['ate']:>+8.4f}"
          f"{row['ate_se']:>8.4f}{row['ate_ci_low']:>+10.4f}"
          f"{row['ate_ci_high']:>+10.4f}{int(row['n']):>8,}"
          f"{row['cate_positive_share']*100:>7.1f}%")


In [ ]:
# â”€â”€ 2Ã—3 CATE distribution grid (publication style) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.linewidth": 0.7, "axes.edgecolor": "#333333",
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
NAVY, RUST = "#1f3b63", "#a6531c"

_specs = [
    ("Revenues",    cate_r1, ate_r1, cate_r2, ate_r2),
    ("Value added", cate_v1, ate_v1, cate_v2, ate_v2),
    ("Employment",  cate_e1, ate_e1, cate_e2, ate_e2),
]

fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2), constrained_layout=True)

for col, (name, c1, a1, c2, a2) in enumerate(_specs):
    for row, (carr, ad, hz, clr) in enumerate([
        (c1, a1, "$t{+}1$ (2023)", NAVY),
        (c2, a2, "$t{+}2$ (2024)", RUST),
    ]):
        ax = axes[row, col]
        ax.hist(carr, bins=50, color=clr, alpha=0.85,
                edgecolor="white", linewidth=0.2, zorder=2)
        ax.axvline(0, color="#555555", lw=0.8, ls=(0, (4, 3)), zorder=1)
        ax.axvline(carr.mean(), color="#444444", lw=1.3, zorder=3)
        ax.text(0.97, 0.93, f"ATE = {ad['ate']:+.3f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor="#cccccc", linewidth=0.5))
        ax.text(0.03, 0.93, f"{name}\n{hz}",
                transform=ax.transAxes, ha="left", va="top", fontsize=8.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if row == 1:
            ax.set_xlabel("Estimated CATE")
        if col == 0:
            ax.set_ylabel("Frequency")

fig.savefig(OUT_DIR / "fig_cate_comparison.pdf", bbox_inches="tight")
plt.close(fig)
print("CATE comparison figure saved.")

In [ ]:
# Combined SHAP beeswarm figures for publication
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.linewidth": 0.7, "axes.edgecolor": "#333333",
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

def plot_combined_shap_beeswarm(specs, filename):
    """Plot three SHAP beeswarms side by side using paper-ready feature labels."""
    fig, axes = plt.subplots(1, len(specs), figsize=(13.2, 5.0), sharex=False)
    if len(specs) == 1:
        axes = [axes]

    for ax, (label, panel_title) in zip(axes, specs):
        payload = SHAP_PLOT_DATA[label]
        plt.sca(ax)
        shap.summary_plot(
            payload['shap'].values,
            payload['features'].values,
            feature_names=payload['display_names'],
            max_display=12,
            show=False,
            color_bar=False,
            plot_size=None,
        )
        ax.set_title(panel_title, fontsize=10, pad=8)
        ax.set_xlabel("SHAP contribution to estimated CATE")
        ax.tick_params(axis="y", labelsize=8)

    # Shared color cue for standardized feature values used by the model.
    fig.text(0.985, 0.72, "High", ha="right", va="center", fontsize=8, color="#b2182b")
    fig.text(0.985, 0.30, "Low", ha="right", va="center", fontsize=8, color="#2166ac")
    fig.text(0.985, 0.50, "Standardized feature value", ha="right", va="center",
             fontsize=8, rotation=90, color="#333333")
    fig.tight_layout(rect=(0, 0, 0.965, 1))
    fig.savefig(OUT_DIR / filename, bbox_inches="tight")
    plt.close(fig)
    print(f"Combined SHAP beeswarm saved: {filename}")

plot_combined_shap_beeswarm(
    [
        ("Revenues t+1 (2023)", "Log operating revenues"),
        ("VA t+1 (2023)", "IHS value added"),
        ("Employment t+1 (2023)", "Log employment"),
    ],
    "fig_shap_beeswarm_t1.pdf",
)

plot_combined_shap_beeswarm(
    [
        ("Revenues t+2 (2024)", "Log operating revenues"),
        ("VA t+2 (2024)", "IHS value added"),
        ("Employment t+2 (2024)", "Log employment"),
    ],
    "fig_shap_beeswarm_t2.pdf",
)


In [ ]:
# â”€â”€ Rebuild the SHAP inputs needed for waterfalls, matching compute_shap â”€â”€â”€â”€â”€â”€
def shap_waterfall_inputs(model, X, feature_names):
    """Reproduce compute_shap's subsample + surrogate to recover Xs, CATEs,
    SHAP values and the explainer base value (needed for additive waterfalls)."""
    rng = np.random.default_rng(42)                       # same seed as compute_shap
    idx = rng.choice(len(X), size=min(SHAP_MAX, len(X)), replace=False)
    Xs  = X[idx]
    cs  = np.squeeze(model.effect(Xs))
    surr = _GBR(n_estimators=300, max_depth=3, learning_rate=0.05,
                min_samples_leaf=20, random_state=42)      # same surrogate
    surr.fit(Xs, cs)
    expl = shap.TreeExplainer(surr)
    sv   = expl.shap_values(Xs, check_additivity=False)
    shap_df = pd.DataFrame(sv, columns=feature_names)
    X_vals  = pd.DataFrame(Xs, columns=feature_names)
    base    = float(np.squeeze(expl.expected_value))
    return shap_df, X_vals, np.asarray(cs), base


In [ ]:
# model_r1 is the CausalForestDML returned by run_forest for revenues t+1
shap_r1, X_vals_r1, cs_r1, base_r1 = shap_waterfall_inputs(mod_r1, X_r1, ALL_COVS)

In [ ]:
# â”€â”€ SHAP waterfall plots for example firms (publication style) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.linewidth": 0.7, "axes.edgecolor": "#333333",
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
NAVY, RUST = "#1f3b63", "#a6531c"

SHAP_DF = shap_r1
X_VALS  = X_vals_r1
EFFECT  = cs_r1
BASE    = base_r1
TOP_K   = 10

# Feature labels are defined centrally in the helpers cell.
# Dummy columns are displayed without numeric suffixes.

order = np.argsort(EFFECT)
examples = {
    "Low-effect firm":    order[int(0.05 * len(order))],
    "Median-effect firm": order[int(0.50 * len(order))],
    "High-effect firm":   order[int(0.95 * len(order))],
}

def _disp_name(name, fvals):
    disp = plot_feature_label(name)
    if name.startswith(DUMMY_PREFIXES):
        return disp                                   # categorical: no value suffix
    if name in fvals.index:
        return disp
    return disp

def draw_waterfall(ax, phi, fvals, base, label):
    s = phi.reindex(phi.abs().sort_values(ascending=False).index)
    top = s.iloc[:TOP_K]
    other = s.iloc[TOP_K:].sum()
    if abs(other) > 0:
        top = pd.concat([top, pd.Series({f"{len(s)-TOP_K} other features": other})])
    top = top.iloc[::-1]

    labels = [_disp_name(n, fvals) for n in top.index]
    vals   = list(top.values)

    cum = base; tips = []
    for i, v in enumerate(vals):
        ax.barh(i, v, left=cum, color=(NAVY if v >= 0 else RUST),
                height=0.62, edgecolor="white", linewidth=0.4, zorder=3)
        tips.append(cum + v); cum += v
    pred = base + sum(vals)

    lo = min([base, pred] + tips); hi = max([base, pred] + tips)
    pad = 0.16 * (hi - lo if hi > lo else abs(hi) + 1e-3)
    ax.set_xlim(lo - pad, hi + pad)
    dx = 0.012 * (ax.get_xlim()[1] - ax.get_xlim()[0])

    cum = base
    for i, v in enumerate(vals):
        ax.text(cum + v + (dx if v >= 0 else -dx), i, f"{v:+.3f}", va="center",
                ha="left" if v >= 0 else "right", fontsize=7, color="#333333", zorder=4)
        cum += v

    ax.axvline(base, color="#888888", lw=0.9, ls=(0, (4, 3)), zorder=1)
    ax.axvline(pred, color="#111111", lw=1.0, zorder=2)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_ylim(-1.6, len(labels) + 0.3)
    ax.text(base, len(labels) + 0.05, f"E[$\\hat{{\\theta}}$]={base:+.3f}",
            ha="center", va="bottom", fontsize=7.5, color="#555555")
    ax.text(pred, -1.35, f"$\\hat{{\\theta}}$={pred:+.3f}",
            ha="center", va="bottom", fontsize=8, color="#111111")
    ax.set_xlabel("Estimated CATE contribution")
    ax.set_title(label, fontsize=9, pad=14)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(axis="y", length=0)

fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.2), constrained_layout=True)
for ax, (label, idx) in zip(axes, examples.items()):
    draw_waterfall(ax, SHAP_DF.iloc[idx], X_VALS.iloc[idx], BASE, label)

fig.savefig(OUT_DIR / "fig_shap_waterfall.pdf", bbox_inches="tight")
plt.close(fig)
print("Waterfall figure saved.")


# Copy publication figures to the LaTeX figure directory.
from shutil import copy2
FIGURE_DIR = PROJECT_ROOT / 'manuscript' / 'latex' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PUBLICATION_FIGURES = [
    'fig_ps_overlap.pdf',
    'fig_event_study_combined.pdf',
    'fig_cate_comparison.pdf',
    'fig_shap_beeswarm_t1.pdf',
    'fig_shap_beeswarm_t2.pdf',
    'fig_shap_waterfall.pdf',
]
for fig_name in PUBLICATION_FIGURES:
    src = OUT_DIR / fig_name
    if src.exists():
        copy2(src, FIGURE_DIR / fig_name)
print(f"Publication figures synced to: {FIGURE_DIR}")


In [ ]:
from sklearn.model_selection import cross_val_predict

def ovb_sensitivity(Y, T, X, label, cf=0.03):
    """Long-Story-Short OVB sensitivity bound (Chernozhukov et al. 2022) for the
    partially-linear DML estimand. cf = benchmark partial-R^2 a confounder would
    explain of BOTH residual outcome and residual treatment variation."""
    m = cross_val_predict(HIST_Y, X, Y, cv=5)
    g = cross_val_predict(HIST_T, X, T, cv=5, method='predict_proba')[:, 1]
    Yr, Tr = Y - m, T - g
    theta = np.sum(Tr * Yr) / np.sum(Tr * Tr)
    eps   = Yr - theta * Tr
    sigma2 = np.mean(eps**2)
    nu2    = 1.0 / np.mean(Tr**2)
    S      = np.sqrt(sigma2 * nu2)                       # scale of the worst-case bias
    se     = np.std(Tr * eps) / (np.mean(Tr**2) * np.sqrt(len(Y)))
    Cy = Cd = np.sqrt(cf / (1 - cf))
    bias = S * Cy * Cd
    rv = abs(theta) / (S + abs(theta))                   # robustness value
    print(f'\n  --- OVB sensitivity [{label}] ---')
    print(f'  theta (PLR)         = {theta:+.4f}  (approx SE {se:.4f})')
    print(f'  bias bound (cf={cf:.0%})  = {bias:.4f}   ->  theta-bias = {theta-bias:+.4f}')
    print(f'  robustness value RV = {rv:.3f}  '
          f'(a confounder must explain >{rv:.1%} of both residual variations to flip the sign)')
    return {'theta': float(theta), 'se': float(se),
            'bias': float(bias), 'rv': float(rv), 'cf': cf}

In [ ]:
SENS = {}
SENS['Revenues t+1']   = ovb_sensitivity(Y_r1, T_r1, X_r1, 'Revenues t+1')
SENS['Revenues t+2']   = ovb_sensitivity(Y_r2, T_r2, X_r2, 'Revenues t+2')
SENS['VA t+1']         = ovb_sensitivity(Y_v1, T_v1, X_v1, 'VA t+1')
SENS['VA t+2']         = ovb_sensitivity(Y_v2, T_v2, X_v2, 'VA t+2')
SENS['Employment t+1'] = ovb_sensitivity(Y_e1, T_e1, X_e1, 'Employment t+1')
SENS['Employment t+2'] = ovb_sensitivity(Y_e2, T_e2, X_e2, 'Employment t+2')

pd.DataFrame.from_dict(SENS, orient='index').reset_index(names='specification').to_csv(OUT_DIR / 'table_sensitivity.csv', index=False)


In [ ]:
from sklearn.neighbors import NearestNeighbors

def psm_att(Y, T, X, label, caliper=0.2):
    """1:1 nearest-neighbour matching on the logit propensity score, with a
    caliper of `caliper` SDs of the logit-PS. Reports the ATT on the matched set."""
    ps = cross_val_predict(HIST_T, X, T, cv=5, method='predict_proba')[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)
    logit = np.log(ps / (1 - ps))
    cal = caliper * np.std(logit)
    t_idx, c_idx = np.where(T == 1)[0], np.where(T == 0)[0]
    nn = NearestNeighbors(n_neighbors=1).fit(logit[c_idx].reshape(-1, 1))
    dist, pos = nn.kneighbors(logit[t_idx].reshape(-1, 1))
    keep = dist.ravel() <= cal
    diffs = Y[t_idx[keep]] - Y[c_idx[pos.ravel()[keep]]]
    att = diffs.mean()
    se  = diffs.std(ddof=1) / np.sqrt(len(diffs))
    print(f'\n  --- PSM 1:1 ATT [{label}] ---')
    print(f'  matched pairs = {len(diffs):,}  ({keep.mean():.1%} of treated within caliper)')
    print(f'  ATT = {att:+.4f}  SE = {se:.4f}  '
          f'95%CI [{att-1.96*se:+.4f}, {att+1.96*se:+.4f}]')
    return {'att': float(att), 'se': float(se), 'n_pairs': int(len(diffs))}

In [ ]:
def did_2x2(Y_post, Y_pre, T, label):
    """Canonical 2x2 difference-in-differences."""
    dY = Y_post - Y_pre
    tr, co = dY[T == 1], dY[T == 0]
    did = tr.mean() - co.mean()
    se  = np.sqrt(tr.var(ddof=1)/len(tr) + co.var(ddof=1)/len(co))
    print(f'\n  --- DiD 2x2 [{label}] ---')
    print(f'  Delta treated = {tr.mean():+.4f}   Delta control = {co.mean():+.4f}')
    print(f'  DiD = {did:+.4f}  SE = {se:.4f}  '
          f'95%CI [{did-1.96*se:+.4f}, {did+1.96*se:+.4f}]')
    return {'did': float(did), 'se': float(se)}

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# Matched difference-in-differences (manuscript robustness estimator)
# ╚══════════════════════════════════════════════════════════════════════════╝
# Pairs each treated firm 1:1 with a control firm on the logit propensity score
# (caliper in SDs), then contrasts their pre->post CHANGE. Differencing removes
# any time-invariant level imbalance left after matching, which is why this is
# the estimator reported in the robustness table rather than a level-only PSM.
from sklearn.neighbors import NearestNeighbors


def matched_did_att(Y_post, Y_pre, T, X, label, caliper=0.2):
    ps = np.clip(
        cross_val_predict(HIST_T, X, T, cv=5, method='predict_proba')[:, 1],
        1e-6, 1 - 1e-6)
    logit = np.log(ps / (1 - ps))
    cal = caliper * np.std(logit)
    t_idx, c_idx = np.where(T == 1)[0], np.where(T == 0)[0]
    nn = NearestNeighbors(n_neighbors=1).fit(logit[c_idx].reshape(-1, 1))
    dist, pos = nn.kneighbors(logit[t_idx].reshape(-1, 1))
    keep = dist.ravel() <= cal
    tk, ck = t_idx[keep], c_idx[pos.ravel()[keep]]
    lvl = Y_post[tk] - Y_post[ck]                       # level-only PSM (for reference)
    chg = (Y_post[tk] - Y_pre[tk]) - (Y_post[ck] - Y_pre[ck])   # matched DiD
    out = {
        'psm_levels':     float(lvl.mean()),
        'psm_levels_se':  float(lvl.std(ddof=1) / np.sqrt(len(lvl))),
        'matched_did':    float(chg.mean()),
        'matched_did_se': float(chg.std(ddof=1) / np.sqrt(len(chg))),
        'n_pairs':        int(len(chg)),
    }
    print(f'\n  --- Matched DiD [{label}] ---')
    print(f'  pairs={out["n_pairs"]:,}  '
          f'PSM(levels)={out["psm_levels"]:+.4f}  '
          f'Matched DiD={out["matched_did"]:+.4f} (SE {out["matched_did_se"]:.4f})')
    return out


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# PHASE 5 — ROBUSTNESS: causal forest vs matched DiD vs two-period DiD
# ╚══════════════════════════════════════════════════════════════════════════╝
# Reproduces Table "Average effects under three estimation strategies".
MDID, DID = {}, {}
_m = master.set_index('nif')

# ---------- REVENUES (log1p, same scale pre and post) ----------
print('\n' + '=' * 65 + '\nROBUSTNESS — Revenues t+1\n' + '=' * 65)
_sub = _m.loc[nifs_r1]
MDID['Revenues t+1'] = matched_did_att(Y_r1, _sub['rev_2021'].values, T_r1, X_r1, 'Revenues t+1')
DID['Revenues t+1']  = did_2x2(_sub['log_rev_2023'].values, _sub['rev_2021'].values,
                               _sub['treated'].values, 'Revenues t+1')

print('\n' + '=' * 65 + '\nROBUSTNESS — Revenues t+2\n' + '=' * 65)
_sub = _m.loc[nifs_r2]
MDID['Revenues t+2'] = matched_did_att(Y_r2, _sub['rev_2021'].values, T_r2, X_r2, 'Revenues t+2')
DID['Revenues t+2']  = did_2x2(_sub['log_rev_2024'].values, _sub['rev_2021'].values,
                               _sub['treated'].values, 'Revenues t+2')

# ---------- EMPLOYMENT (log1p) ----------
print('\n' + '=' * 65 + '\nROBUSTNESS — Employment t+1\n' + '=' * 65)
_sub = _m.loc[nifs_e1]
MDID['Employment t+1'] = matched_did_att(Y_e1, _sub['log_emp_2021'].values, T_e1, X_e1, 'Employment t+1')
DID['Employment t+1']  = did_2x2(_sub['log_emp_2023'].values, _sub['log_emp_2021'].values,
                                 _sub['treated'].values, 'Employment t+1')

print('\n' + '=' * 65 + '\nROBUSTNESS — Employment t+2\n' + '=' * 65)
_sub = _m.loc[nifs_e2]
MDID['Employment t+2'] = matched_did_att(Y_e2, _sub['log_emp_2021'].values, T_e2, X_e2, 'Employment t+2')
DID['Employment t+2']  = did_2x2(_sub['log_emp_2024'].values, _sub['log_emp_2021'].values,
                                 _sub['treated'].values, 'Employment t+2')

# ---------- VALUE ADDED (post = log(VA>0); pre = log(VA_2021>0), same rows) ----------
print('\n' + '=' * 65 + '\nROBUSTNESS — VA t+1\n' + '=' * 65)
_va21 = np.sinh(_m.loc[nifs_v1, 'ihs_va_2021'].values)
_ok = _va21 > 0
MDID['VA t+1'] = matched_did_att(Y_v1[_ok], np.log(_va21[_ok]), T_v1[_ok], X_v1[_ok], 'VA t+1')
DID['VA t+1']  = did_2x2(Y_v1[_ok], np.log(_va21[_ok]), T_v1[_ok], 'VA t+1')

print('\n' + '=' * 65 + '\nROBUSTNESS — VA t+2\n' + '=' * 65)
_va21 = np.sinh(_m.loc[nifs_v2, 'ihs_va_2021'].values)
_ok = _va21 > 0
MDID['VA t+2'] = matched_did_att(Y_v2[_ok], np.log(_va21[_ok]), T_v2[_ok], X_v2[_ok], 'VA t+2')
DID['VA t+2']  = did_2x2(Y_v2[_ok], np.log(_va21[_ok]), T_v2[_ok], 'VA t+2')

# ---------- assemble the manuscript robustness table ----------
_forest = {
    'Revenues t+1': ate_r1, 'Revenues t+2': ate_r2,
    'VA t+1': ate_v1, 'VA t+2': ate_v2,
    'Employment t+1': ate_e1, 'Employment t+2': ate_e2,
}
robustness_rows = []
for spec in ['Revenues t+1', 'Revenues t+2', 'VA t+1', 'VA t+2',
             'Employment t+1', 'Employment t+2']:
    robustness_rows.append({
        'specification':   spec,
        'forest_ate':      _forest[spec]['ate'],
        'forest_se':       _forest[spec]['ate_se'],
        'matched_did':     MDID[spec]['matched_did'],
        'matched_did_se':  MDID[spec]['matched_did_se'],
        'did':             DID[spec]['did'],
        'did_se':          DID[spec]['se'],
        'psm_levels':      MDID[spec]['psm_levels'],   # reference: inflated levels contrast
        'n_pairs':         MDID[spec]['n_pairs'],
    })
pd.DataFrame(robustness_rows).to_csv(OUT_DIR / 'table_robustness.csv', index=False)
print('\n\nDone. Forest / Matched DiD / DiD written to table_robustness.csv.')
